# Romi — NeuroMatch 2026 · Posterior Motives

**Your slide's questions:**

- Does it recover the **learned prior width per block**? (abstract claim ii)
- Is each subject actually **learning**? (the learning-rate parameter λ)
- Prior learning **across coherences**; ANOVA (coherence + prior_std).
- (Behavioral learning excluded for now. Share code with Salma.)

This notebook is runnable top-to-bottom and uses **HB-Adaptive** (our hierarchical Bayesian observer) and the **Switch** (paper's switching observer). Edit `SUBJECT` / filters and re-run any cell. See the API guide cell for how to make your own calls.

In [1]:
# ============================================================
#  SETUP  —  works in Google Colab or on a local checkout
# ============================================================
import os, sys, subprocess

REPO_URL = "https://github.com/salmaelhanchi/NeuroMatch_2026_Behaviour.git"
BRANCH   = "model-verification"     # branch that carries the fitted models + API

def _find_root():
    """If we're already inside a checkout, use it (no clone needed)."""
    here = os.getcwd()
    for _ in range(6):
        if os.path.isfile(os.path.join(here, "observers", "api.py")):
            return here
        here = os.path.dirname(here)
    return None

ROOT = _find_root()
if ROOT is None:
    # Colab path: shallow-clone the repo, then point at the hierarchical/ package.
    dest = "/content/NeuroMatch_2026_Behaviour" if os.path.isdir("/content") \
           else os.path.abspath("NeuroMatch_2026_Behaviour")
    if not os.path.isdir(dest):
        # PUBLIC repo: this just works. PRIVATE repo: replace REPO_URL with
        #   https://<TOKEN>@github.com/salmaelhanchi/NeuroMatch_2026_Behaviour.git
        subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1",
                        REPO_URL, dest], check=True)
    ROOT = os.path.join(dest, "hierarchical")

sys.path.insert(0, ROOT)
os.chdir(ROOT)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from observers import api

print("repo root :", ROOT)
print("models    :", api.list_models())
print("data      :", "data/data01_direction4priors.csv  (12 subjects)")

# figures for this notebook go in a dedicated subfolder beside it
FIG_DIR = os.path.join(ROOT, "experiments", "romi", "01_slide_notebook", "figures")
os.makedirs(FIG_DIR, exist_ok=True)
print("figures  :", FIG_DIR)

repo root : /content/NeuroMatch_2026_Behaviour/hierarchical
models    : ['switch', 'basic_bayes', 'hb_adaptive', 'hb_rachel', 'hb_salma', 'recombined', 'hierarchical_online', 'reliability_mixture']
data      : data/data01_direction4priors.csv  (12 subjects)
figures  : /content/NeuroMatch_2026_Behaviour/hierarchical/experiments/romi/01_slide_notebook/figures


## How to use the model API (read me — also for an LLM assistant)

Everything below comes from the **fitted models**, reached through one module:
`observers.api`. You never touch raw model math — you call functions.

### Where the data lives (relative to the repo's `hierarchical/` folder)
| what | path |
|---|---|
| trial-level data (all 12 subjects) | `data/data01_direction4priors.csv` |
| point fits (per model, per subject) | `results/fits/comparison/<model>/subject<N>.json` |
| cross-validation records | `results/fits/comparison_cv/<model>/subject<N>_cv.json` |
| project abstract | `docs/project_abstract.md` |

### Model keys
`'switch'` (paper's Switching observer, 9 params, non-learning) · `'hb_adaptive'`
(our hierarchical Bayesian observer, 6 params, **learns** prior width AND mixing
weight α online) · also available: `'hb_rachel'`, `'hb_salma'`, `'recombined'`,
`'basic_bayes'`.

### The API — five kinds of call
```python
from observers import api

# LOAD --------------------------------------------------------------
api.load_subject(2)                 # -> DataFrame: motion_direction, motion_coherence,
                                    #    prior_std, estimate_dir  (one subject's trials)
api.load_fitted('hb_adaptive', 2)     # -> (observer, record) with fitted params
api.observed_distribution(2, direction=335, coherence=0.06, prior_std=80)
                                    # -> empirical response histogram (density over 1..360)

# INSPECT -----------------------------------------------------------
api.list_models(); api.model_info() # what exists, params, colors
api.fitted_subjects('hb_adaptive')    # which subjects are fit

# PREDICT -----------------------------------------------------------
api.predict('hb_adaptive', 2)         # -> (n_trials, 360) predicted distribution per trial
api.belief_trajectory('hb_adaptive', 2)
                                    # -> DataFrame trial, believed_sd  (the LEARNED prior width)

# FIT / SIMULATE ----------------------------------------------------
api.fit_model('hb_adaptive', 2)       # refit from scratch (slow)
api.simulate('hb_adaptive', 2, seed=0)# generative: synthetic responses from the fitted model

# EVALUATE ----------------------------------------------------------
api.results_table()                 # -> tidy DataFrame: model,label,subject,k,nll,aic,bic,cv_nll
api.trial_logliks('hb_adaptive', 2)   # -> per-trial log-likelihood (slice it however you like)
api.bias_variability(2)             # -> per-condition estimation bias + circular SD (Fig-3 core)
```

### Custom calls (when a helper doesn't exist)
The API returns raw numbers; **you** aggregate/plot. To reach the model object
directly:
```python
from observers.comparison.registry import build_registry, load_subject
spec = build_registry(['hb_adaptive'])['hb_adaptive']   # a ModelSpec
obs, rec = api.load_fitted('hb_adaptive', 2)
dists = spec.predict_distributions(obs, load_subject(2))  # (n_trials, 360)
```
Condition labels for any trial-level array come from `api.load_subject(s)` —
they're **row-aligned** with `predict`, `trial_logliks`, and `belief_trajectory`.

## 1 · Learned belief per trial (shared with Salma)

Same building block as Salma's notebook: the fitted observer's learned prior width per trial, joined with condition labels.

In [ ]:
# ---- shared helper: learned belief joined with per-trial conditions ----
# belief_trajectory() replays the FITTED observer over the subject's real trials
# and returns the learned prior width (believed_sd) per trial. We join it with
# the condition labels (coherence, prior_std), which are ROW-ALIGNED with it.
def belief_with_conditions(key, subject):
    bt = api.belief_trajectory(key, subject)          # trial, believed_sd
    d  = api.load_subject(subject).reset_index(drop=True)
    out = bt.copy()
    out['coherence'] = d['motion_coherence'].values
    out['prior_std'] = d['prior_std'].values
    out['subject']   = subject
    return out

def all_beliefs(key, subjects=range(1,13)):
    return pd.concat([belief_with_conditions(key, s) for s in subjects],
                     ignore_index=True)

allb = all_beliefs('hb_adaptive')
print('rows:', len(allb), '| columns:', list(allb.columns))

## 2 · Does it recover the learned prior width per block?

The belief trajectory for one subject, with the true block structure overlaid — the direct read of abstract claim (ii).

In [ ]:
S = 2   # edit: example subject
b = belief_with_conditions('hb_adaptive', S)
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(b.trial, b.believed_sd, color='#d1495b', lw=1.3, label='believed prior SD (learned)')
ax.step(b.trial, b.prior_std, where='post', color='k', alpha=0.5, lw=1, label='true block SD')
ax.set_xlabel('trial'); ax.set_ylabel('prior SD (deg)')
ax.set_title(f'subject {S}: HB-Adaptive recovers block-specific prior width online')
ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, 'romi_belief_trajectory.png'), dpi=130); plt.show()

### 2b · Companion plot — learned confidence in the prior (α)

HB-Adaptive learns a second quantity online, alongside the prior width: **α**, how much it trusts the prior each trial (`believed_alpha`, already in `b` from the cell above — same subject, same trial axis).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(b.trial, b.believed_alpha, color='#2e86ab', lw=1.3, label='believed α (confidence in the prior)')
ax.axhline(0.5, color='k', ls=':', alpha=0.5, lw=1, label='α = 0.5 (equal weight)')
ax.set_xlabel('trial'); ax.set_ylabel('α (prior confidence, 0–1)')
ax.set_ylim(-0.02, 1.02)
ax.set_title(f'subject {S}: HB-Adaptive learned confidence in the prior (α) over trials')
ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, 'romi_alpha_trajectory.png'), dpi=130); plt.show()

## 3 · Is each subject actually learning? — the learning-rate λ

λ (`lam`) is HB-Rachel's forgetting / learning-rate parameter. λ>0 means the belief is updated from feedback (the learning machinery is engaged). We read the fitted λ for every subject.

In [ ]:
import json
lam = {}
for s in range(1,13):
    lam[s] = json.load(open(f'results/fits/comparison/hb_adaptive/subject{s}.json'))['params']['lam']
lam = pd.Series(lam, name='lam')
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(lam.index, lam.values, color='#d1495b')
ax.set_xlabel('subject'); ax.set_ylabel('fitted learning rate  λ')
ax.set_title('Every subject has λ>0 — the learning machinery is engaged for all')
ax.set_xticks(range(1,13))
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, 'romi_learning_rate.png'), dpi=130); plt.show()
print('lambda by subject:'); print(lam.round(3).to_string())
print(f'\nrange: {lam.min():.3f} - {lam.max():.3f}  (all > 0 => all subjects learning)')

## 4 · In which prior blocks did the model learn best?

Recovery accuracy per block = how close the learned width is to the true block SD. Smaller absolute error = better recovery.

In [ ]:
true_sd = {10:10, 20:20, 40:40, 80:80}
rec = allb.groupby(['prior_std']).believed_sd.mean()
err = pd.DataFrame({'true': pd.Series(true_sd), 'learned': rec.round(2)})
err['abs_error'] = (err['learned'] - err['true']).abs().round(2)
print('Recovery of prior width by block:'); print(err.to_string())
print('\nNarrow blocks (10/20) are recovered tightly; the 80 block compresses')
print('(learned < true) — wide priors are harder to pin down from finite feedback.')

## 5 · Prior learning across coherences + ANOVA (shared with Salma)

In [ ]:
# ---- shared helper: two-way ANOVA believed_sd ~ coherence + prior_std ----
# Uses PER-SUBJECT cell means (not raw trials) to avoid pseudoreplication.
import statsmodels.api as sm
import statsmodels.formula.api as smf
def anova_believed_sd(df):
    cell = df.groupby(['subject','coherence','prior_std']).believed_sd.mean().reset_index()
    model = smf.ols('believed_sd ~ C(coherence) + C(prior_std)', data=cell).fit()
    aov = sm.stats.anova_lm(model, typ=2)
    aov['eta_sq'] = aov['sum_sq'] / aov['sum_sq'].sum()
    return aov[['sum_sq','F','PR(>F)','eta_sq']].round(4)

# learned width by prior_std, one line per coherence
fig, ax = plt.subplots(figsize=(6.6, 4.4))
colors = {0.06:'#ffb3c1', 0.12:'#ff5c8a', 0.24:'#c9184a'}
for coh in [0.06,0.12,0.24]:
    m = allb[np.isclose(allb.coherence,coh)].groupby('prior_std').believed_sd.mean()
    ax.plot(m.index, m.values, 'o-', color=colors[coh], label=f'coh={coh}')
ax.plot([10,20,40,80],[10,20,40,80],'k:',alpha=0.5,label='true')
ax.set_xlabel('nominal prior SD'); ax.set_ylabel('learned believed SD')
ax.set_title('Learned width vs block width — overlaps across coherence'); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, 'romi_across_coherence.png'), dpi=130); plt.show()
print(anova_believed_sd(allb).to_string())
print('\nComparing across coherences at matched prior_std: the lines overlap =>')
print('coherence does NOT change the learned width (only prior_std does).')